# 01 · Async concurrency for LLM calls

Make ten LLM calls in the time of one, cap how many run at once, stop the slow ones, stream results as they
finish and keep the event loop responsive: the concurrency toolkit behind every LLM service.

**Time:** ~25 min · **Runs:** offline on CPU in < 1 min · **Needs:** [async/await basics](../01_python_basics/14_async_await_basics.ipynb)

## Why this matters in interviews

- An LLM call is almost pure **waiting**. Running independent calls concurrently is the cheapest latency win there
  is: same model, same data, often 5-10x faster wall time. "Parallelise everything independent" is on every
  list of RAG latency fixes.
- "Why would you use async for an LLM endpoint?" is a standard question. The follow-ups are the traps: a blocking
  call inside `async def`, unbounded fan-out that turns into a wall of 429s, timeouts that never fire usefully.
- At scale the ceiling is the provider's quota, not your CPU. Knowing **Little's law** (in-flight = throughput x
  latency) lets you size a worker pool on a whiteboard.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `tl08` | Why would you use async for an endpoint that calls an LLM? |
| `in29` | What is the difference between concurrency, batch size and throughput? |
| `po41` | Give me eight fixes for a slow RAG pipeline that change neither the data nor the model. |
| `po01` | Your system works for 100 users. What changes at 100,000? |
| `po17` | Why are timeouts more delicate in an LLM system than in a normal service? |
| `in26` | How should you handle an LLM timeout? |
| `in27` | What is request hedging and what does it cost? |
| `po20` | Why put a queue in front of a production GenAI system? |
| `pd11` | What is async processing and when should an LLM feature be asynchronous? |

## Setup

The first cell finds `llm_setup.py` and gives us a real `openai.OpenAI` client (OpenAI, Ollama or the offline
stand-in). Offline, the stand-in answers instantly, so we switch on its `latency` knob: every request then takes
0.3 s, like a short real API call. With a real provider the latency is simply real.

In [ ]:
import sys
from pathlib import Path
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "llm_setup.py").exists():
        sys.path.insert(0, str(_p)); break
from llm_setup import client, MODEL, EMBED_MODEL, PROVIDER

In [ ]:
import asyncio
import math
import time
from concurrent.futures import ThreadPoolExecutor

import matplotlib.pyplot as plt
import numpy as np
import openai
from llm_setup import OFFLINE, make_async_client

aclient = make_async_client()          # AsyncOpenAI: same API, but every call is awaitable

# Warm-up: the very first call pays one-off import costs inside the SDK. Never time it.
client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": "hi"}], max_tokens=5)
await aclient.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": "hi"}], max_tokens=5)

LATENCY = 0.3
if OFFLINE:                            # offline only: make each request take 0.3 s
    OFFLINE.latency = LATENCY
N = 10 if OFFLINE else 5               # fewer calls when a real provider bills you
QUESTIONS = [f"What is {i}*7?" for i in range(1, N + 1)]
print(f"provider={PROVIDER}  model={MODEL}  calls per experiment={N}")

## 1. An LLM call is waiting, not working

**In plain English:** calling a model is like ordering at a busy restaurant. Almost all of the time is the
kitchen, not you. Measure it: wall-clock time versus the CPU time this process actually used.

In [ ]:
def ask(question: str) -> str:
    r = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": question}], max_tokens=20)
    return r.choices[0].message.content

wall0, cpu0 = time.perf_counter(), time.process_time()
answer = ask("What is 6*7?")
wall, cpu = time.perf_counter() - wall0, time.process_time() - cpu0
print(f"answer: {answer!r}")
print(f"wall time: {wall:.3f} s   CPU time used by this process: {cpu:.3f} s")
assert cpu < wall / 2, "an LLM call is I/O-bound: we mostly wait"

That is the whole argument for concurrency: while one request waits, the process could be sending the next
nine. The work is **I/O-bound** (waiting on a network), not CPU-bound, so threads or `asyncio` both work and
the GIL is irrelevant.

## 2. Three ways to make N calls: sequential, threads, `asyncio.gather`

In [ ]:
# (a) sequential: one after another
t0 = time.perf_counter()
seq_answers = [ask(q) for q in QUESTIONS]
t_seq = time.perf_counter() - t0

# (b) threads: the *sync* client in a thread pool (works with any blocking library)
t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=N) as pool:
    thread_answers = list(pool.map(ask, QUESTIONS))      # map keeps input order
t_threads = time.perf_counter() - t0

# (c) asyncio: the async client, all requests in flight at once
async def aask(question: str) -> str:
    r = await aclient.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": question}],
                                              max_tokens=20)
    return r.choices[0].message.content

t0 = time.perf_counter()
async_answers = await asyncio.gather(*(aask(q) for q in QUESTIONS))   # top-level await works in Jupyter
t_gather = time.perf_counter() - t0

print(f"sequential : {t_seq:.2f} s\nthreads    : {t_threads:.2f} s\ngather     : {t_gather:.2f} s")
print("first answers:", async_answers[:2])
assert seq_answers == thread_answers == list(async_answers), "same answers, same order"
assert t_gather < t_seq / 2 and t_threads < t_seq / 2, "concurrency should be several times faster"

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.2))
bars = ax.bar(["sequential\nfor loop", "threads\n(sync client)", "asyncio.gather\n(async client)"],
              [t_seq, t_threads, t_gather], color=["#adb5bd", "#74c0fc", "#3b5bdb"])
ax.bar_label(bars, fmt="%.2f s")
ax.set_ylabel("wall time (s)")
ax.set_title(f"{N} LLM calls: sequential pays every wait, concurrent pays roughly one")
plt.show()

Sequential time is about `N x latency`; concurrent time is about **one** latency. Both concurrent versions give the
answers back in input order (`pool.map` and `gather` both preserve order).

| | Threads (`ThreadPoolExecutor` + sync SDK) | `asyncio` (async SDK) |
|---|---|---|
| Works with | any blocking library (sync DB driver, `requests`) | libraries that offer `async` APIs (`AsyncOpenAI`, `httpx.AsyncClient`) |
| How many in flight | tens to low hundreds (each thread costs memory and scheduling) | thousands in one thread; the limit becomes the provider quota |
| Cancelling a slow call | not really (you cannot kill a thread) | built in: `task.cancel()`, `asyncio.timeout()` |
| Classic bug | shared mutable state without a lock | one blocking call stalls **every** request (section 8) |

**Gotcha:** `asyncio.run(main())` raises `RuntimeError: asyncio.run() cannot be called from a running event loop`
in Jupyter, because Jupyter already runs a loop. In a notebook, `await main()` at top level; in a script, `asyncio.run(main())`.

## 3. `gather` and errors: `return_exceptions`, and `TaskGroup`

By default the first exception raised inside `gather` propagates to you, **and the other calls keep running**
in the background. With `return_exceptions=True` each failure comes back as a value in its slot, so a fan-out
can return partial results. Offline we make the stand-in fail one request with a 500.

In [ ]:
fragile = make_async_client(max_retries=0)       # no SDK retries, so the failure reaches us
if OFFLINE:
    OFFLINE.fail_next(1, 500)                    # the next request fails with HTTP 500

async def aask_with(c, question, max_tokens=60):
    r = await c.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": question}],
                                        max_tokens=max_tokens)
    return r.choices[0].message.content

results = await asyncio.gather(*(aask_with(fragile, q) for q in QUESTIONS[:3]), return_exceptions=True)
for q, r in zip(QUESTIONS[:3], results):
    print(f"{q:<14} -> {type(r).__name__ if isinstance(r, Exception) else repr(r)}")
errors = [r for r in results if isinstance(r, Exception)]
if OFFLINE:
    assert len(errors) == 1 and isinstance(errors[0], openai.InternalServerError)
print(f"{len(results) - len(errors)} answers kept, {len(errors)} failure(s) handled")

`asyncio.TaskGroup` (Python 3.11+) has the opposite, **all-or-nothing** semantics: when one task fails it cancels
its siblings and raises an `ExceptionGroup`. Use it when a partial result is useless (all three parts of a
report are needed); use `gather(..., return_exceptions=True)` when partial results are fine (search lanes).

In [ ]:
async def fake_llm_call(latency: float = 0.1, result: str = "ok") -> str:
    """Stands in for an LLM request: pure waiting, no CPU. Free and deterministic with any provider."""
    await asyncio.sleep(latency)
    return result

cancelled = []
async def part(name, latency, fail=False):
    try:
        await asyncio.sleep(latency)
        if fail:
            raise ValueError(f"{name} failed")
        return name
    except asyncio.CancelledError:
        cancelled.append(name)        # record, then re-raise: never swallow CancelledError
        raise

t0 = time.perf_counter()
try:
    async with asyncio.TaskGroup() as tg:
        for name, lat, fail in [("intro", 1.0, False), ("body", 0.1, True), ("summary", 1.0, False)]:
            tg.create_task(part(name, lat, fail))
except* ValueError as eg:
    print("TaskGroup raised:", [str(e) for e in eg.exceptions])
print(f"siblings cancelled: {sorted(cancelled)} after {time.perf_counter() - t0:.2f} s (not 1 s)")
assert sorted(cancelled) == ["intro", "summary"]

## 4. Cap concurrency with `asyncio.Semaphore`

`gather` over 10,000 prompts would open 10,000 requests at once: instant 429s from the provider, exhausted
connection pools and memory. A **semaphore** is a counter of permits: at most `limit` coroutines hold one at a
time, the rest wait. We sweep the limit for 20 calls of 0.1 s (fake calls, so the sweep is free with any provider)
and also track the peak number in flight to prove the cap holds.

In [ ]:
async def run_bounded(n_calls: int, limit: int, latency: float = 0.1):
    sem = asyncio.Semaphore(limit)
    in_flight = peak = 0
    async def one(i):
        nonlocal in_flight, peak
        async with sem:                         # waits here when `limit` calls are already running
            in_flight += 1; peak = max(peak, in_flight)
            try:
                return await fake_llm_call(latency)
            finally:
                in_flight -= 1
    t0 = time.perf_counter()
    await asyncio.gather(*(one(i) for i in range(n_calls)))
    return time.perf_counter() - t0, peak

N_CALLS, CALL_S = 20, 0.1
limits = [1, 2, 4, 5, 10, 20]
sweep = {L: await run_bounded(N_CALLS, L, CALL_S) for L in limits}
for L, (t, peak) in sweep.items():
    print(f"limit={L:>2}: {t:.2f} s   peak in flight={peak:>2}   throughput={N_CALLS / t:5.1f} req/s")
    assert peak == L, "the semaphore caps concurrency exactly"
times = [sweep[L][0] for L in limits]
assert all(a > b for a, b in zip(times, times[1:])), "more permits, less wall time"

In [ ]:
ideal = [math.ceil(N_CALLS / L) * CALL_S for L in limits]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].plot(limits, times, "o-", label="measured")
axes[0].plot(limits, ideal, "k--", alpha=0.6, label="ideal: ceil(N / limit) x latency")
axes[0].set(xlabel="semaphore limit (max in flight)", ylabel="wall time (s)", title="20 calls x 0.1 s")
axes[0].legend()
axes[1].plot(limits, [N_CALLS / t for t in times], "o-", color="#2f9e44", label="measured")
axes[1].plot(limits, [L / CALL_S for L in limits], "k--", alpha=0.6, label="Little's law: limit / latency")
axes[1].set(xlabel="semaphore limit (max in flight)", ylabel="throughput (req/s)", title="Throughput = concurrency / latency")
axes[1].legend()
plt.tight_layout(); plt.show()

**Little's law** is the one formula to remember: `in-flight requests = throughput x latency`. With latency fixed
by the model, the only way to raise throughput is more requests in flight, until something else caps you: the
provider quota, a connection pool, or the GPU's batch capacity on a self-hosted server (then latency starts
rising, the "knee" of `in29`). A semaphore sized from the quota is how you stay below that knee.

## 5. Timeouts, deadlines and cancellation

A request that never returns holds a connection and a slot in your semaphore forever. Put a timeout on every
await that crosses the network. `asyncio.timeout()` (3.11+) cancels the awaited task when time runs out and
works with any client.

In [ ]:
t0 = time.perf_counter()
try:
    async with asyncio.timeout(LATENCY / 2):             # budget shorter than the call takes
        await aask("What is 2+2?")
    timed_out = False
except TimeoutError:
    timed_out = True
print(f"timed out: {timed_out} after {time.perf_counter() - t0:.2f} s")
if OFFLINE:
    assert timed_out

The SDK has its own knobs too: `make_async_client(timeout=20.0, max_retries=2)` or per call
`aclient.with_options(timeout=5.0).chat.completions.create(...)`. The OpenAI SDK's default timeout is 600 s, far
longer than any user waits. (The offline stand-in has no real socket, so only `asyncio.timeout` fires here.)

**Deadline propagation** (`po17`): a request has *one* budget. Each step should get the time that is left, not a
fixed per-call timeout; otherwise a slow first step makes the whole request blow its budget. Below, a 1.2 s budget
covers retrieval (0.3 s) and rerank (0.5 s), so generation gets only what remains and the pipeline degrades
instead of running late.

In [ ]:
class Deadline:
    def __init__(self, seconds): self.end = time.perf_counter() + seconds
    def remaining(self): return max(0.0, self.end - time.perf_counter())

async def step(name, latency, deadline, log):
    budget = deadline.remaining()
    try:
        async with asyncio.timeout(budget):
            await fake_llm_call(latency)
        log.append(f"{name:<9} done      ({latency:.1f} s needed, {budget:.2f} s left)")
        return True
    except TimeoutError:
        log.append(f"{name:<9} CANCELLED ({latency:.1f} s needed, only {budget:.2f} s left)")
        return False

deadline, log = Deadline(1.2), []
ok = [await step(n, s, deadline, log) for n, s in [("retrieve", 0.3), ("rerank", 0.5), ("generate", 1.0)]]
print("\n".join(log))
print("response:", "full answer" if all(ok) else "degraded: return the retrieved sources, marked as partial")
assert ok == [True, True, False]

**Hedging** (`in27`) uses the same cancellation machinery: if the answer has not arrived by roughly the p95
latency, fire a duplicate and keep whichever finishes first, cancelling the other. Only calls slower than the
hedge threshold trigger one, so hedging at p95 costs about 5% more calls (this toy has a heavier tail, so more).
Only hedge **idempotent** requests (reads, not refunds).

In [ ]:
rng = np.random.default_rng(0)
n_req = 20
primary_s = np.where(rng.random(n_req) < 0.15, 1.0, 0.1)      # 15% of calls hit a slow replica
backup_s = np.full(n_req, 0.1)

async def hedged(p_s, b_s, hedge_after=0.2):
    first = asyncio.create_task(fake_llm_call(p_s))
    done, _ = await asyncio.wait({first}, timeout=hedge_after)
    if done:
        return False                                          # no hedge needed
    second = asyncio.create_task(fake_llm_call(b_s))
    done, pending = await asyncio.wait({first, second}, return_when=asyncio.FIRST_COMPLETED)
    for t in pending:
        t.cancel()                                            # stop paying for the loser
    return True

async def timed(coro):
    t0 = time.perf_counter(); out = await coro
    return time.perf_counter() - t0, out

plain = await asyncio.gather(*(timed(fake_llm_call(p)) for p in primary_s))
hedge = await asyncio.gather(*(timed(hedged(p, b)) for p, b in zip(primary_s, backup_s)))
lat_plain, lat_hedge = [t for t, _ in plain], [t for t, _ in hedge]
print(f"no hedging : p50={np.percentile(lat_plain, 50):.2f} s  max={max(lat_plain):.2f} s")
print(f"hedged     : p50={np.percentile(lat_hedge, 50):.2f} s  max={max(lat_hedge):.2f} s  "
      f"extra calls={sum(h for _, h in hedge)}/{n_req}")
assert max(lat_hedge) < max(lat_plain) / 2

## 6. Stream results as they finish: `as_completed`

`gather` waits for the slowest call before you see anything. `asyncio.as_completed` hands you each result the
moment it is ready: show the first search lane's hits, the first candidate answer, the first finished document.
Here five calls with different latencies (fake calls, so the order is deterministic).

In [ ]:
jobs = {"doc-A": 0.5, "doc-B": 0.1, "doc-C": 0.3, "doc-D": 0.2, "doc-E": 0.4}
spans, order = {}, []
T0 = time.perf_counter()

async def summarise(name, latency):
    start = time.perf_counter() - T0
    await fake_llm_call(latency)
    spans[name] = (start, time.perf_counter() - T0)
    return name

for fut in asyncio.as_completed([summarise(n, s) for n, s in jobs.items()]):
    name = await fut
    order.append(name)
    print(f"{time.perf_counter() - T0:.2f} s  got {name}")
assert order == sorted(jobs, key=jobs.get), "results arrive fastest-first, not in submission order"

fig, ax = plt.subplots(figsize=(8, 2.6))
for i, name in enumerate(jobs):
    s, e = spans[name]
    ax.barh(i, e - s, left=s, color="#74c0fc", edgecolor="#1c7ed6")
    ax.text(e + 0.01, i, f"#{order.index(name) + 1} to finish", va="center", fontsize=8)
ax.set_yticks(range(len(jobs)), list(jobs)); ax.invert_yaxis()
ax.set_xlim(0, 1.2 * max(e for _, e in spans.values()))
ax.set(xlabel="seconds since start", title="as_completed: all start together, results handed over as each finishes")
plt.show()

## 7. Fan-out / fan-in (map-reduce)

**In plain English:** split the work, send the pieces out at the same time, then combine. Summarising a long
document: summarise each chunk concurrently (fan-out), then summarise the summaries (fan-in). Total time is about
two latencies, however many chunks there are (as long as the semaphore/quota allows).

In [ ]:
document = [
    "The billing service moved to a new database in March. Migration took two weekends.",
    "Latency on the search page fell by forty percent after the cache was added. Users noticed.",
    "The support team now answers refund questions with the assistant. Escalations dropped.",
    "Next quarter the team plans to add a second model provider. A fallback drill is scheduled.",
]

async def map_reduce_summary(chunks, limit=8):
    sem = asyncio.Semaphore(limit)
    async def summarise_chunk(text):
        async with sem:
            return await aask_with(aclient, f"Summarise this in one sentence.\n\n{text}")
    partials = await asyncio.gather(*(summarise_chunk(c) for c in chunks))      # fan-out
    notes = " ".join(p.removeprefix("Summary:").strip() for p in partials)      # drop a label if the model added one
    final = await aask_with(aclient, f"Summarise these notes in two sentences.\n\n{notes}")   # fan-in
    return partials, final

t0 = time.perf_counter()
partials, final = await map_reduce_summary(document)
t_mr = time.perf_counter() - t0
for p in partials:
    print("  partial:", p)
print("final:", final)
print(f"{len(document)} chunks + 1 combine in {t_mr:.2f} s (sequential would be ~{(len(document) + 1) * LATENCY:.1f} s offline)")
if OFFLINE:
    assert t_mr < 3 * LATENCY

The offline stand-in "summarises" by keeping sentences; a real model writes a real summary. The shape is what
matters and it is everywhere: parallel retrieval lanes fused with RRF, guardrail checks run next to generation,
one LLM call per document in a batch job, agents that call several tools at once.

## 8. Never block the event loop

`asyncio` is cooperative: one thread, and a coroutine only gives up control at an `await`. A `time.sleep`, a
**sync** SDK call or a CPU-heavy parse inside `async def` freezes *every* other request on that loop. We make that
visible with a heartbeat coroutine that should wake every 20 ms, and record the longest gap while each job runs.

In [ ]:
async def heartbeat(stop: asyncio.Event, interval: float = 0.02):
    gaps, last = [], time.perf_counter()
    while not stop.is_set():
        await asyncio.sleep(interval)
        now = time.perf_counter(); gaps.append(now - last); last = now
    return gaps

def cpu_work(seconds: float = 0.3) -> int:
    """Pure-Python busy work for a fixed time: stands in for parsing, tokenising, scoring..."""
    end, x = time.perf_counter() + seconds, 0
    while time.perf_counter() < end:
        x += sum(i * i for i in range(300))
    return x

async def longest_gap(job) -> float:
    stop = asyncio.Event()
    hb = asyncio.create_task(heartbeat(stop))
    await asyncio.sleep(0.06)
    await job()
    await asyncio.sleep(0.06)
    stop.set()
    return max(await hb)

async def blocking_sleep(): time.sleep(0.3)                         # BAD: blocks the loop
async def inline_cpu(): cpu_work(0.3)                               # BAD: blocks the loop
async def cpu_to_thread(): await asyncio.to_thread(cpu_work, 0.3)    # OK: runs in a worker thread
async def async_sleep(): await asyncio.sleep(0.3)                   # OK: yields to the loop

gaps = {name: await longest_gap(job) for name, job in
        [("time.sleep in\nasync def", blocking_sleep), ("CPU work in\nasync def", inline_cpu),
         ("asyncio.to_thread\n(CPU work)", cpu_to_thread), ("await\nasyncio.sleep", async_sleep)]}
for name, g in gaps.items():
    print(f"{name.replace(chr(10), ' '):<30} longest heartbeat gap: {1000 * g:6.0f} ms")
g = list(gaps.values())
assert g[0] > 0.25 and g[1] > 0.25, "blocking calls freeze the loop"
assert g[2] < 0.2 and g[3] < 0.1, "to_thread and real awaits keep it responsive"

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2))
bars = ax.bar(list(gaps), [1000 * v for v in gaps.values()], color=["#e03131", "#e03131", "#2f9e44", "#2f9e44"])
ax.bar_label(bars, fmt="%.0f ms")
ax.axhline(20, color="k", ls="--", lw=1, label="expected tick (20 ms)")
ax.set_ylim(0, 1150 * max(gaps.values()))
ax.set(ylabel="longest gap between heartbeats (ms)", title="A blocking call stalls every coroutine on the loop")
ax.legend(); plt.show()

Rules that follow:

- Inside `async def`, only call async libraries (`AsyncOpenAI`, `httpx.AsyncClient`, async DB drivers).
- Stuck with a sync library? `await asyncio.to_thread(fn, *args)` runs it in a thread and frees the loop.
- Genuinely CPU-heavy work (big parses, local embedding models) should go to a process pool
  (`loop.run_in_executor(ProcessPoolExecutor(), fn)`) or a separate worker. `to_thread` keeps the loop responsive,
  but because of the GIL the pure-Python work itself still does not run in parallel.
- [FastAPI serving](04_fastapi_serving_and_streaming.ipynb) measures what this does to a web server under load.

## 9. Concurrency under a rate limit

A semaphore caps **how many are in flight**, not **how many start per second**. Providers limit requests per
minute (RPM) and tokens per minute (TPM). With 0.1 s calls, a semaphore of 5 already starts 50 requests a second,
which would blow a 10 req/s quota. A rate limiter spaces out the starts instead. (A full token bucket, and what to
do with a 429, are in the [next notebook](02_retries_backoff_rate_limits_circuit_breakers.ipynb).)

In [ ]:
class AsyncRateLimiter:
    """At most `rate` request starts per second, evenly spaced. Each caller reserves the next slot."""
    def __init__(self, rate: float):
        self.interval, self.next_slot, self.lock = 1.0 / rate, 0.0, asyncio.Lock()
    async def wait(self):
        async with self.lock:
            now = time.perf_counter()
            slot = max(now, self.next_slot)
            self.next_slot = slot + self.interval
        await asyncio.sleep(slot - now)

async def run(n, rate=None, limit=None, latency=0.1):
    limiter = AsyncRateLimiter(rate) if rate else None
    sem = asyncio.Semaphore(limit or n)
    starts, T = [], time.perf_counter()
    async def one():
        if limiter:
            await limiter.wait()
        async with sem:
            starts.append(time.perf_counter() - T)
            await fake_llm_call(latency)
    await asyncio.gather(*(one() for _ in range(n)))
    return sorted(starts), time.perf_counter() - T

sem_starts, sem_t = await run(20, limit=5)
rl_starts, rl_t = await run(20, rate=10)
print(f"semaphore(5) only : {sem_t:.2f} s -> {20 / sem_t:.0f} req/s   (quota is 10 req/s: 429s incoming)")
print(f"rate limiter 10/s : {rl_t:.2f} s -> {20 / rl_t:.1f} req/s")
assert 20 / sem_t > 15 and 20 / rl_t <= 10.5

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.step(sem_starts, range(1, 21), where="post", label="semaphore(5) only")
ax.step(rl_starts, range(1, 21), where="post", label="rate limiter, 10 req/s")
xs = np.linspace(0, 2, 50)
ax.plot(xs, 10 * xs, "k--", lw=1, label="quota: 10 req/s")
ax.set(xlabel="seconds", ylabel="requests started", title="A semaphore limits concurrency, not rate", ylim=(0, 21))
ax.legend(); plt.show()

**Sizing from the quota** (`po01`): the useful concurrency is `quota_rps x latency`, by Little's law. Beyond that,
extra in-flight requests just queue or get 429s. In production the limiter's state must be **shared** across
replicas (Redis), or ten pods each allowing 10 req/s become 100 req/s.

In [ ]:
def in_flight_needed(throughput_rps: float, latency_s: float) -> float:
    return throughput_rps * latency_s                                   # Little's law

rpm_quota, latency_s = 600, 1.5
print(f"quota {rpm_quota} RPM = {rpm_quota / 60:.0f} req/s; at {latency_s} s per call that is "
      f"~{in_flight_needed(rpm_quota / 60, latency_s):.0f} requests in flight at most")
print(f"to serve 50 req/s at 2 s latency you need {in_flight_needed(50, 2):.0f} in flight")
assert in_flight_needed(rpm_quota / 60, latency_s) == 15

When work takes longer than anyone will watch (a 40 s agent run, a batch of 10,000 documents), the right shape is
not a bigger semaphore but a **queue**: accept, return a job id, and let a bounded worker pool sized from the
quota drain it (`po20`, `pd11`).

## Try it yourself

**1.** Predict, then run: 10 calls of 0.3 s through `run_bounded` with a limit of 3. How long?

In [ ]:
# Solution — try it yourself first, then run this
t, peak = await run_bounded(10, 3, 0.3)
print(f"measured {t:.2f} s, predicted ceil(10/3) x 0.3 = {math.ceil(10 / 3) * 0.3:.1f} s, peak in flight {peak}")
assert abs(t - 1.2) < 0.25

**2.** A classic interview drill: write `bounded_gather(coros, limit)` that runs coroutines with at most `limit`
in flight and returns results **in input order**.

In [ ]:
# Solution — try it yourself first, then run this
async def bounded_gather(coros, limit: int):
    sem = asyncio.Semaphore(limit)
    async def guard(c):
        async with sem:
            return await c
    return await asyncio.gather(*(guard(c) for c in coros))   # gather preserves order

out = await bounded_gather([fake_llm_call(s, result=f"r{i}") for i, s in enumerate([0.3, 0.1, 0.2, 0.1])], limit=2)
print(out)
assert out == ["r0", "r1", "r2", "r3"]

**3.** Fan out 5 fake calls where two are slow (1 s), with a 0.3 s timeout **per call**; keep whatever finished
in time and mark the rest as missing.

In [ ]:
# Solution — try it yourself first, then run this
async def with_timeout(coro, seconds):
    try:
        async with asyncio.timeout(seconds):
            return await coro
    except TimeoutError:
        return None

latencies = [0.1, 1.0, 0.2, 1.0, 0.1]
t0 = time.perf_counter()
got = await asyncio.gather(*(with_timeout(fake_llm_call(s, f"lane{i}"), 0.3) for i, s in enumerate(latencies)))
print(got, f"in {time.perf_counter() - t0:.2f} s")
assert got == ["lane0", None, "lane2", None, "lane4"]

**4.** Your provider gives you 3,000 RPM and calls average 0.8 s. What semaphore limit would you start with, and
what throughput would a limit of 100 actually give you?

In [ ]:
# Solution — try it yourself first, then run this
quota_rps = 3000 / 60
start_limit = in_flight_needed(quota_rps, 0.8)
print(f"quota = {quota_rps:.0f} req/s -> start with ~{start_limit:.0f} in flight")
print(f"limit 100 would *try* {100 / 0.8:.0f} req/s, but the quota caps it at {quota_rps:.0f}: the rest become 429s")

if OFFLINE:                      # reset the stand-in's knobs
    OFFLINE.latency = 0.0

## Say it in an interview

- **30 seconds:** "An LLM call is I/O-bound, so the process mostly waits. With `asyncio` and an async client one
  process holds hundreds of calls in flight; `gather` fans out independent work and `as_completed` streams results
  as they finish. I cap concurrency with a semaphore sized from the provider quota via Little's law, put a
  deadline on the whole request and give each step what is left, and never call blocking code inside
  `async def`: it goes to `to_thread` or a process pool."
- **Numbers:** N calls of latency L take about N x L sequentially and about L concurrently. Throughput =
  concurrency / latency. 600 RPM at 1.5 s needs only ~15 in flight.
- **Threads vs asyncio:** threads for sync libraries and a handful of calls; asyncio for many calls and easy
  cancellation. Both beat sequential because the GIL is released while waiting on I/O.
- **Follow-ups:** "What happens past the knee?" Requests queue and p95 climbs, so admission control or a queue in
  front. "Why not make everything async?" Because one blocking call inside it is worse than a sync endpoint.
  "How do you hedge safely?" Only idempotent requests, cancel the loser.
- **Traps:** `asyncio.run` inside Jupyter; `gather` without `return_exceptions` leaves the other calls running when
  one fails; an unbounded `gather` over 10k prompts; swallowing `CancelledError`; a new client per request
  (throws away the connection pool); a per-call timeout longer than the whole request budget.

## Next

- [02 · Retries, backoff, rate limits, circuit breakers](02_retries_backoff_rate_limits_circuit_breakers.ipynb): what to do when the calls fail.
- [04 · FastAPI serving and streaming](04_fastapi_serving_and_streaming.ipynb): the same event loop inside a web server.
- [05 · Latency percentiles and cost](05_latency_percentiles_p50_p95_and_cost.ipynb): why fan-out amplifies tail latency.
- Background: [async/await basics](../01_python_basics/14_async_await_basics.ipynb), [OpenAI API basics](../08_llm_apps/01_openai_api_basics.ipynb).
- Theory: [AI system design concepts](../../ai_system_design_concepts/index.html) · [interview bank](../../ai_interview_prep/index.html).